In [24]:
import torch
import torch.nn as nn
from model.transformer import Transformer

In [25]:
# English: 2 sentences, 4 tokens each
src_tokens = torch.tensor([
    [15, 87, 204, 319],
    [21, 45, 102, 300]
])

# French: 2 sentences, 3 tokens each
tgt_tokens = torch.tensor([
    [1, 45, 87],
    [1, 62, 103]
])

#Create causal mask for target tokens
tgt_len = tgt_tokens.shape[1]

tgt_mask = torch.triu(
    torch.ones(tgt_len, tgt_len, dtype=torch.bool),
    diagonal=1
)

In [26]:
tgt_mask

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [27]:
# Example French token IDs
# 1 = BOS, 2 = EOS
target_tokens = torch.tensor([
    [1, 45, 87, 2],
    [1, 62, 103, 2]
])

# Decoder input
decoder_input = target_tokens[:, :-1]

# Expected predictions
expected_output = target_tokens[:, 1:]

print("Original:", target_tokens)
print("Decoder input:", decoder_input)
print("Expected output:", expected_output)

Original: tensor([[  1,  45,  87,   2],
        [  1,  62, 103,   2]])
Decoder input: tensor([[  1,  45,  87],
        [  1,  62, 103]])
Expected output: tensor([[ 45,  87,   2],
        [ 62, 103,   2]])


In [28]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [29]:
%pip install datasets tokenizers

Note: you may need to restart the kernel to use updated packages.


In [30]:
from datasets import load_dataset

dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr"
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})


In [31]:
for i in range(5):
    example = dataset["train"][i]
    print(f"Example {i + 1}")
    print("English:", example["translation"]["en"])
    print("French:", example["translation"]["fr"])
    print("---")

Example 1
English: The Wanderer
French: Le grand Meaulnes
---
Example 2
English: Alain-Fournier
French: Alain-Fournier
---
Example 3
English: First Part
French: PREMIÈRE PARTIE
---
Example 4
English: I
French: CHAPITRE PREMIER
---
Example 5
English: THE BOARDER
French: LE PENSIONNAIRE
---


In [32]:
import importlib
import data.tokenizer

importlib.reload(data.tokenizer)

<module 'data.tokenizer' from '/Users/jimmylawson/Desktop/ML Papers Implementations/transformer-from-sratch/data/tokenizer.py'>

In [33]:
from data.tokenizer import english_tokenizer, english_trainer, french_tokenizer, french_trainer


def english_sentence():
    for example in dataset["train"]:
        yield example["translation"]["en"]
english_tokenizer.train_from_iterator(
    english_sentence(),
    trainer=english_trainer,
    length=len(dataset["train"])
)

In [34]:
print(english_tokenizer.get_vocab_size())
english_tokenizer.save("english_tokenizer.json")

8000


In [35]:
sentence = "I love machine learning."
encoding = english_tokenizer.encode(sentence)
print("Original:", sentence)
print("Tokens:", encoding.tokens)
print("Token IDs:", encoding.ids)

Original: I love machine learning.
Tokens: ['I', 'love', 'machine', 'learning', '.']
Token IDs: [42, 736, 6273, 7779, 17]


In [36]:
def french_sentence():
    for example in dataset["train"]:
        yield example["translation"]["fr"]

french_tokenizer.train_from_iterator(
    french_sentence(),
    trainer=french_trainer,
    length=len(dataset["train"])
)

In [37]:
print(french_tokenizer.get_vocab_size())
french_tokenizer.save("french_tokenizer.json")

8000


In [38]:
from data.translation_dataset import TranslationDataset

train_dataset = TranslationDataset(
    dataset= dataset["train"],
    src_tokenizer=english_tokenizer,
    tgt_tokenizer=french_tokenizer,
    max_len=64
)
sample = train_dataset[0]

print("Source tokens:", sample["src_tokens"])
print("Decoder input:", sample["decoder_input"])
print("Expected output:", sample["expected_output"])

Source tokens: tensor([   2,  226,   56,  161,   68, 1877,    3])
Decoder input: tensor([   2,  366,  518, 2170])
Expected output: tensor([ 366,  518, 2170,    3])


In [39]:
from data.masks import create_masks
from data.translation_dataset import collate_fn

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)
batch = next(iter(train_loader))

src_tokens = batch["src_tokens"]
decoder_input = batch["decoder_input"]

src_mask, tgt_mask = create_masks(
    src_tokens,
    decoder_input
)

print("Source tokens:", src_tokens.shape)
print("Decoder input:", decoder_input.shape)

print("Source mask:", src_mask.shape)
print("Target mask:", tgt_mask.shape)

Source tokens: torch.Size([16, 64])
Decoder input: torch.Size([16, 63])
Source mask: torch.Size([16, 1, 1, 64])
Target mask: torch.Size([16, 1, 63, 63])


# 🧠 Transformer: English → French Translation

### Architecture: Encoder–Decoder Transformer
*Implemented from scratch using PyTorch*

---

## 🔹 1. Encoder — Understanding English

**Input:** English token IDs

English Tokens → Token Embeddings → Positional Encoding → Encoder Layers

The encoder processes the English sentence and produces contextual representations.

## 🔹 2. Decoder — Generating French

**Input:** Shifted French token IDs + Encoder Output

French Tokens → Token Embeddings → Positional Encoding → Decoder Layers

The decoder uses:
- **Masked Self-Attention:** Attends only to current and previous French tokens.
- **Cross-Attention:** Attends to the encoder's English representations.
- **Feed-Forward Network:** Transforms each token's representation.

## 🔹 3. Output Projection — Predicting French Tokens

Decoder Output → Linear Projection → Vocabulary Logits

The linear layer converts each decoder hidden state into raw scores for every French vocabulary token.

---

### 📐 Tensor Shapes

| Stage | Shape |
|---|---|
| English tokens | `(B, T_src)` |
| Encoder output | `(B, T_src, d_model)` |
| French decoder input | `(B, T_tgt)` |
| Decoder output | `(B, T_tgt, d_model)` |
| Vocabulary logits | `(B, T_tgt, V_fr)` |

**Training objective:** Predict the next French token using cross-entropy loss.

**Hardware:** Apple Silicon GPU (MPS)

In [40]:
from model.transformer import Transformer

src_vocab_size = english_tokenizer.get_vocab_size()
tgt_vocab_size = french_tokenizer.get_vocab_size()

model = Transformer(
    src_vocab_size,
    tgt_vocab_size,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    dropout=0.1
).to(device)

#we use french padding here because the model predict french tokens
pad_id = french_tokenizer.token_to_id("[PAD]")


optimizer = torch.optim.Adam(model.parameters(), lr=0.0001,betas=(0.9,0.98),eps=1e-9 )

In [41]:
model.train()
#Get one batch
batch = next(iter(train_loader))

src_tokens = batch["src_tokens"].to(device)
decoder_input = batch["decoder_input"].to(device)
expected_output = batch["expected_output"].to(device)

print("Source shape:",src_tokens.shape)
print("Decoder input shape:",decoder_input.shape)
print("Expected output shape:",expected_output.shape)


Source shape: torch.Size([16, 55])
Decoder input shape: torch.Size([16, 51])
Expected output shape: torch.Size([16, 51])


In [42]:
criterion = nn.CrossEntropyLoss(
    ignore_index=pad_id # dont want to include padding in loss
)



In [43]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        #move tensor to MPS
        src_tokens = batch["src_tokens"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        expected_output = batch["expected_output"].to(device)
        #Create source and target masks
        src_mask, tgt_mask = create_masks(
             src_tokens,
            decoder_input,
        pad_id=1
        )
         #Forward pass
        logits = model(src_tokens, decoder_input, src_mask, tgt_mask)

        #calculate loss for this batch
        loss = criterion(
            logits.reshape(-1,tgt_vocab_size),
            expected_output.reshape(-1)
        )
        #Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    #Avg loss
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: Loss = {avg_loss:.4f}")




Epoch 1: Loss = 4.7098
Epoch 2: Loss = 3.8560
Epoch 3: Loss = 3.5154
Epoch 4: Loss = 3.2867
Epoch 5: Loss = 3.1104
Epoch 6: Loss = 2.9621
Epoch 7: Loss = 2.8355
Epoch 8: Loss = 2.7266
Epoch 9: Loss = 2.6303
Epoch 10: Loss = 2.5443


In [44]:

print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)

MPS available: True
CUDA available: False
Model device: mps:0


In [45]:
torch.save(
    {
    "epoch":10,
    "model_state_dict":model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "training_loss": 2.5443,
    },
    "transformer_checkpoint.pth"
)
print("Transformer checkpoint saved successfully!")

Transformer checkpoint saved successfully!


In [48]:
torch.save(
    model.state_dict(),
    "transformer_weights.pth"
)
print("Transformer weights saved successfully!")

Transformer weights saved successfully!


In [46]:
from pathlib import Path

for filename in [
    "transformer_checkpoint.pth",
]:
    path = Path(filename)
    print(f"{filename}: {path.exists()}")

transformer_checkpoint.pth: True


In [49]:
weights = torch.load(
    "transformer_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(name, tensor.shape)

encoder.embedding.embedding.weight torch.Size([8000, 512])
encoder.pos_encoding.p_e torch.Size([5000, 512])
encoder.pos_encoding.embedding.weight torch.Size([5000, 512])
encoder.layers.0.attention.W_q.weight torch.Size([512, 512])
encoder.layers.0.attention.W_q.bias torch.Size([512])
encoder.layers.0.attention.W_k.weight torch.Size([512, 512])
encoder.layers.0.attention.W_k.bias torch.Size([512])
encoder.layers.0.attention.W_v.weight torch.Size([512, 512])
encoder.layers.0.attention.W_v.bias torch.Size([512])
encoder.layers.0.attention.W_o.weight torch.Size([512, 512])
encoder.layers.0.attention.W_o.bias torch.Size([512])
encoder.layers.0.ffn.linear1.weight torch.Size([2048, 512])
encoder.layers.0.ffn.linear1.bias torch.Size([2048])
encoder.layers.0.ffn.linear2.weight torch.Size([512, 2048])
encoder.layers.0.ffn.linear2.bias torch.Size([512])
encoder.layers.0.residual1.norm.weight torch.Size([512])
encoder.layers.0.residual1.norm.bias torch.Size([512])
encoder.layers.0.residual2.norm.w

In [50]:
model.load_state_dict(
      torch.load(
        "transformer_weights.pth",
        map_location=device,
        weights_only=True
    )
)
model = model.to(device)
model.eval()

print("model loaded successfully!")
print("Device:", next(model.parameters()).device)


model loaded successfully!
Device: mps:0


In [51]:
from tokenizers import Tokenizer
english_tokenizer = Tokenizer.from_file("english_tokenizer.json")
french_tokenizer = Tokenizer.from_file("french_tokenizer.json")

In [52]:
#Verify
sentence = "I love machine learning."

original_ids = english_tokenizer.encode(sentence).ids
loaded_tokenizer = Tokenizer.from_file("english_tokenizer.json")
loaded_ids = loaded_tokenizer.encode(sentence).ids

print("Original:", original_ids)
print("Loaded:", loaded_ids)

print("Same?", original_ids == loaded_ids)

Original: [42, 736, 6273, 7779, 17]
Loaded: [42, 736, 6273, 7779, 17]
Same? True


In [53]:
bos_id = french_tokenizer.token_to_id("[BOS]")
eos_id = french_tokenizer.token_to_id("[EOS]")
pad_id = french_tokenizer.token_to_id("[PAD]")

print("BOS:", bos_id)
print("EOS:", eos_id)
print("PAD:", pad_id)

BOS: 2
EOS: 3
PAD: 1


In [54]:
@torch.no_grad()
def greedy_decode(
    model,
    sentence,
    english_tokenizer,
    french_tokenizer,
    device,
    max_len=64
):
    model.eval()
    # 1. Encode English sentence
    src_ids = english_tokenizer.encode(sentence).ids
    src_ids = (
        [english_tokenizer.token_to_id("[BOS]")]
        + src_ids
        + [english_tokenizer.token_to_id("[EOS]")]
    )
    src_tokens = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )
    # 2. Start decoder with [BOS]
    bos_id = french_tokenizer.token_to_id("[BOS]")
    eos_id = french_tokenizer.token_to_id("[EOS]")
    decoder_input = torch.tensor(
        [[bos_id]],
        dtype=torch.long,
        device=device
    )
    # 3. Generate tokens one at a time
    for _ in range(max_len):
        # Source padding mask
        src_mask = (
            src_tokens == english_tokenizer.token_to_id("[PAD]")
        ).unsqueeze(1).unsqueeze(2)
        # Causal mask
        tgt_len = decoder_input.size(1)
        tgt_mask = torch.triu(
            torch.ones(
                tgt_len,
                tgt_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        ).unsqueeze(0).unsqueeze(0)
        # 4. Forward pass
        logits = model(
            src_tokens,
            decoder_input,
            src_mask,
            tgt_mask
        )
        # 5. Select the last position
        next_token_logits = logits[:, -1, :]

        # 6. Choose highest-scoring token
        next_token = torch.argmax(
            next_token_logits,
            dim=-1,
            keepdim=True
        )
        # 7. Append prediction
        decoder_input = torch.cat(
            [decoder_input, next_token],
            dim=1
        )

        # 8. Stop at EOS
        if next_token.item() == eos_id:
            break

    # 9. Convert predicted token IDs to French text
    predicted_ids = decoder_input[0].tolist()

    translation = french_tokenizer.decode(
        predicted_ids,
        skip_special_tokens=True
    )

    return translation

In [57]:
sentence = "I love God."
translation = greedy_decode(
    model,
    sentence,
    english_tokenizer,
    french_tokenizer,
    device
)

print("English:", sentence)
print("French:", translation)

English: I love God.
French: Je aime .


In [56]:
encoded = french_tokenizer.encode("étudiant")

print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)
print("Decoded:", french_tokenizer.decode(encoded.ids))

Tokens: ['étudi', 'ant']
IDs: [5738, 190]
Decoded: étudi ant


In [58]:
print("English:", english_tokenizer.encode("God").tokens)
print("French:", french_tokenizer.encode("Dieu").tokens)

English: ['God']
French: ['Dieu']


In [59]:
sentences = [
    "I love you.",
    "I love my mother.",
    "I love God.",
    "God loves me.",
    "I am happy."
]

for sentence in sentences:
    translation = greedy_decode(
        model,
        sentence,
        english_tokenizer,
        french_tokenizer,
        device
    )

    print(f"{sentence} → {translation}")

I love you. → Je t ’ aime .
I love my mother. → Je aime ma mère .
I love God. → Je aime .
God loves me. → Dieu m ' aime .
I am happy. → Je suis heureuse .


In [60]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Millions of parameters: {total_params / 1e6:.2f}M")

Total parameters: 61,554,496
Trainable parameters: 61,554,496
Millions of parameters: 61.55M
